# Site-level run coverage by area (rough)
A rough share of the world's protected area that the site-level global run evaluated.

- **Evaluated area**: the area of the 12,759 evaluated sites (sites in `global_results/`, as in `site_coverage.ipynb`), from the cleaned site geometries the run used (`global_results/site_geoms.parquet`), flattened so overlaps count once. Area is geodesic on the WGS84 ellipsoid (`pa_area.flat_area_km2`).
- **Total PA area**: Protected Planet's Nov 2026 coverage of terrestrial and inland waters by protected areas (not OECMs), 17.51% of land, from [protectedplanet.net](https://www.protectedplanet.net/en) ("Statistics updated: Nov 2026"; read on 2026-11-24). The page gives no area, so the share is converted to km² with the land area implied by the [Protected Planet Report 2024](https://digitalreport.protectedplanet.net/) (Aug 2024 data): its 17.58% coverage by PAs and OECMs is 22.06 + 1.59 million km² ("284,242 protected areas cover a total of 22.06 million km2"; "6,253 OECMs cover an additional 1.59 million km2").

Why this is rough:
- Protected Planet clips PAs to land with its base map. The evaluated area isn't clipped, so the sea parts of coastal sites count, which overstates the share.
- Site geometries are simplified to 15 m.
- The land area comes from the 2024 report, not the Nov 2026 statistics.

In [1]:
from pathlib import Path
import os
import sys

# Show only gRPC errors (gcsfs loads gRPC, which logs each worker process start)
os.environ["GRPC_VERBOSITY"] = "ERROR"

import ee
import geopandas as gpd
import numpy as np
import pandas as pd

# Repo root: nearest folder up from here with pyproject.toml
REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO / "src"))

from cells.get_control_cells import init_ee
from data_analysis.pa_area import flat_area_km2, geodesic_area_km2
from sites.chunker import wdpa_features
from utils.retries import with_retries
from utils.variables import PROJECT, PAS_ASSET_ID, SITE_CHUNKS, GLOBAL_RESULTS_PREFIX

# Protected Planet Report 2024 (Aug 2024 data), terrestrial and inland waters
PP_2024_PA_KM2 = 22.06e6  # protected areas
PP_2024_OECM_KM2 = 1.59e6  # OECMs
PP_2024_COVERAGE = 0.1758  # protected areas and OECMs, share of land
# protectedplanet.net, statistics updated Nov 2026: protected areas, share of land
PP_2026_PA_COVERAGE = 0.1751

## Total PA area

In [2]:
# Land area implied by the 2024 report
land_km2 = (PP_2024_PA_KM2 + PP_2024_OECM_KM2) / PP_2024_COVERAGE
pp_2026_pa_km2 = PP_2026_PA_COVERAGE * land_km2
print(f"Land area implied by the 2024 report: {land_km2:,.0f} km²")

Land area implied by the 2024 report: 134,527,873 km²


## Evaluated area

In [3]:
evaluated = set(pd.read_csv(f"{GLOBAL_RESULTS_PREFIX}relative_results.csv")["site_id"])
site_geoms = gpd.read_parquet(f"{GLOBAL_RESULTS_PREFIX}site_geoms.parquet")
evaluated_geoms = site_geoms[site_geoms["site_id"].isin(evaluated)]
assert set(evaluated_geoms["site_id"]) == evaluated
assert evaluated_geoms["site_id"].is_unique
assert evaluated_geoms.crs.to_epsg() == 4326

evaluated_km2 = flat_area_km2(evaluated_geoms.geometry)
# Sum of each site's area, with overlaps counted more than once
unflattened_km2 = evaluated_geoms.geometry.apply(geodesic_area_km2).sum()
assert evaluated_km2 <= unflattened_km2
print(f"Unflattened sum: {unflattened_km2:,.0f} km²")

Unflattened sum: 20,543,028 km²


## Check: marine share of the evaluated area
How much clipping the evaluated area to land could change the result. WDPA reports each polygon's total area (`GIS_AREA`) and its marine part (`GIS_M_AREA`), both computed by UNEP-WCMC. These are summed over the evaluated sites' polygons that `clean_sites.py` fetched (`chunker.wdpa_features`), from the run's WDPA release. Removing the same marine share from the flattened area is only approximate, because overlaps are counted in the WDPA sums.

In [4]:
chunks = pd.read_parquet(SITE_CHUNKS)
run_release = chunks["wdpa_update_time"].iloc[0]
init_ee(PROJECT)
current_release = ee.data.getAsset(PAS_ASSET_ID)["updateTime"]
assert current_release == run_release, (
    f"WDPA is now {current_release}; the run used {run_release}"
)

# Evaluated sites' polygons in wdpa_features, attributes only
features = (
    wdpa_features()
    .filter(ee.Filter.inList("SITE_ID", [int(s) for s in sorted(evaluated)]))
    .select(["SITE_ID", "GIS_AREA", "GIS_M_AREA"], None, False)
)
wdpa_areas = with_retries(
    ee.data.computeFeatures,
    {"expression": features, "fileFormat": "PANDAS_DATAFRAME"},
    label="WDPA areas",
)
site_areas = wdpa_areas.groupby("SITE_ID")[["GIS_AREA", "GIS_M_AREA"]].sum()
assert set(site_areas.index) == evaluated
# GIS_AREA sums match the chunk table's
chunk_areas = chunks.set_index("site_id").loc[site_areas.index, "gis_area_km2"]
assert np.allclose(site_areas["GIS_AREA"], chunk_areas)

marine_share = site_areas["GIS_M_AREA"].sum() / site_areas["GIS_AREA"].sum()
print(f"Marine share of evaluated sites' WDPA area: {marine_share:.1%}")
print(
    "Share of PA area evaluated with that marine share removed: "
    f"{evaluated_km2 * (1 - marine_share) / pp_2026_pa_km2:.1%}"
)

Marine share of evaluated sites' WDPA area: 2.7%
Share of PA area evaluated with that marine share removed: 71.0%


## Summary

In [5]:
print(f"Evaluated PA area (flattened):                 {evaluated_km2:,.0f} km²")
print(f"Protected Planet PA area, Nov 2026 (derived):  {pp_2026_pa_km2:,.0f} km²")
print(
    f"Share of PA area evaluated:                    {evaluated_km2 / pp_2026_pa_km2:.1%}"
)

Evaluated PA area (flattened):                 17,180,265 km²
Protected Planet PA area, Nov 2026 (derived):  23,555,830 km²
Share of PA area evaluated:                    72.9%
